# IT494: a document, unit by unit
#
Each unit of a document (here a chapter of a public-domain novel; a chat session works
the same) becomes entities, quote-backed facts, a summary, and per-entity cells, in its
own workspace with no identity resolved. Every fact carries a verbatim quote; a quote
that is not in the text is dropped and counted. Then, with the whole document in
evidence, the major entities are reconciled bottom up, a store is built over the major
entities alone, the predicate vocabulary is consolidated, and the graph is drawn.
Nothing in the prompts knows what a novel is; the only corpus-specific cell is the loader.
#
To run: attach the units dataset, add an `OPENAI_API_KEY` secret, turn Internet on, Run All.
#
## 1 · The units

In [ ]:
import json, re, time
from pathlib import Path

DATA = next(Path("/kaggle/input").rglob("units.jsonl")).parent
documents = [json.loads(l) for l in (DATA / "documents.jsonl").read_text(encoding="utf-8").splitlines()]
all_units = [json.loads(l) for l in (DATA / "units.jsonl").read_text(encoding="utf-8").splitlines()]

DOC = "oz/01_55.txt"
doc = next(d for d in documents if d["source_uri"] == DOC)
units = sorted((u for u in all_units if u["doc_id"] == doc["doc_id"]), key=lambda u: u["position"])
print(f"{DOC}: {len(units)} units, {sum(len(u['text']) for u in units)} chars")
for u in units:
    print(f"  {u['label']:<14} {len(u['text']):>6} chars")

## 2 · The model
One function makes every call and logs its cost. One gate checks that a string the model
claims to have copied is really in the unit.

In [ ]:
import requests
from kaggle_secrets import UserSecretsClient

MODEL = "gpt-5.6-luna"                # extraction
JUDGE = "gpt-5.6-terra"               # reconciliation and vocabulary verdicts
PRICE = {"gpt-5.6-luna": (0.20, 1.20), "gpt-5.6-terra": (2.00, 12.00)}   # $ per M tokens in, out
SPEND_STOP = 8.00                     # dollars; the run halts past this
KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
calls = []

def spend():
    return sum(c["cost"] for c in calls)

def generate(prompt, stage, model=MODEL, effort="low"):
    """One JSON-mode call; the reply parsed, the cost logged."""
    if spend() >= SPEND_STOP:
        raise RuntimeError(f"spending stop: ${spend():.2f}")
    t0 = time.time()
    r = requests.post("https://api.openai.com/v1/chat/completions",
                      headers={"Authorization": f"Bearer {KEY}"}, timeout=180,
                      json={"model": model, "reasoning_effort": effort,
                            "response_format": {"type": "json_object"},
                            "messages": [{"role": "user", "content": prompt}]})
    if r.status_code != 200:
        raise RuntimeError(f"OpenAI {r.status_code}: {r.text[:300]}")
    body = r.json()
    u, (p_in, p_out) = body["usage"], PRICE[model]
    calls.append({"stage": stage, "model": body["model"], "in": u["prompt_tokens"],
                  "out": u["completion_tokens"], "seconds": round(time.time() - t0, 1),
                  "cost": (u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out) / 1e6})
    return json.loads(body["choices"][0]["message"]["content"])

def in_text(text, s):
    """The gate: the string occurs in the unit, line wraps forgiven."""
    return bool(s.split()) and re.search(r"\s+".join(re.escape(w) for w in s.split()), text) is not None

## 3 · The prompts
Entities, then facts about them, then the summary and cells. Each prompt starts with the
unit text so the three calls share a cached prefix.

In [ ]:
def entity_prompt(unit):
    return f"""TEXT ({unit['label']}):
{unit['text']}

Above is one unit of a longer document. Identify the entities it involves: each person, group, place, thing, event, or topic that acts, is acted upon, or is discussed in its own right, plus any named person, place, group, or thing, however briefly mentioned. Parts, components, and possessions of a listed entity are not entities; they belong inside that entity's facts.

Return JSON {{"entities": [{{"name", "named", "kind", "salience", "surface_forms"}}]}} where:
- name: for a named entity, the fullest name the text uses; for an unnamed one, a head word plus a parenthetical anchoring it to a named entity, like "car (Sam's car)"
- named: true if the text gives it a proper name
- kind: one lowercase word: person, group, place, object, event, topic, or another if none fits
- salience: "major" only if it would appear in a two-sentence summary of this text, else "minor"
- surface_forms: every distinct verbatim string the text uses to refer to it, bare pronouns excluded"""

def fact_prompt(unit, names, majors):
    return f"""TEXT ({unit['label']}):
{unit['text']}

Above is one unit of a longer document; below, the entities identified in it. For each entity, distill every durable fact the text states about it: what it is, its attributes, its states, its situation, its parts and possessions, its relationships to the other listed entities. Not moment-to-moment actions or passing remarks; a lasting disposition, habit, or position counts.

Return JSON {{"facts": [{{"subject", "predicate", "object", "qualifiers", "quote"}}]}} where:
- subject: a name from ENTITIES, exactly as written
- predicate: a lowercase_snake_case relation in the present tense, named the way the text gives it
- object: another entity's name exactly as written when the fact relates two entities, else a short literal value; never a bare true or false
- qualifiers: a short phrase for role, manner, or condition, else null
- quote: one verbatim substring of the text that supports the fact, copied exactly
- each fact is atomic: one attribute or one relationship, with a bare value rather than a phrase
- a relationship between a major entity and a minor one is stated once, under the major entity; only between two major entities may it be stated from both sides

ENTITIES: {", ".join(names)}
MAJOR: {", ".join(majors)}"""

def cells_prompt(unit, named):
    return f"""TEXT ({unit['label']}):
{unit['text']}

Above is one unit of a longer document. Write:
1. "summary": the unit in 3-5 concrete sentences.
2. "cells": for each entity below that appears, 1-3 sentences in third person on what it does, what happens to it, or what is learned about it in this unit, written so it can be appended to that entity's running record.
Use only what the text says.

Return JSON {{"summary": "...", "cells": [{{"entity", "text"}}]}}

ENTITIES: {", ".join(named)}"""

## 4 · Derive one unit
Three calls, two gates, one record per unit. Nothing is shared between units.

In [ ]:
def derive(unit):
    text = unit["text"]
    entities, dropped = [], []
    for e in generate(entity_prompt(unit), "entities")["entities"]:
        e["surface_forms"] = [s for s in e["surface_forms"] if in_text(text, s)]
        (entities if e["surface_forms"] else dropped).append(e)
    names = [e["name"] for e in entities]
    majors = [e["name"] for e in entities if e["salience"] == "major"]

    facts, lost = [], []
    for f in generate(fact_prompt(unit, names, majors), "facts")["facts"]:
        if f["subject"] in names and in_text(text, f["quote"]):
            f["object_is_entity"] = f["object"] in names
            facts.append(f)
        else:
            lost.append(f)

    out = generate(cells_prompt(unit, [e["name"] for e in entities if e["named"]]), "cells")
    cells = [c for c in out["cells"] if c["entity"] in names]
    return {"unit": unit["label"], "position": unit["position"], "entities": entities,
            "dropped": dropped, "facts": facts, "lost": lost, "summary": out["summary"], "cells": cells}

def show(rec, full=False):
    print("=" * 88)
    print(f"{rec['unit']}: {len(rec['entities'])} entities, {len(rec['facts'])} facts, "
          f"{len(rec['dropped'])} entities and {len(rec['lost'])} facts dropped; run so far ${spend():.3f}")
    for e in rec["entities"]:
        print(f"  {e['name']:<36} {e['kind']:<9} {'named' if e['named'] else 'unnamed':<8} {e['salience']}")
    if not full:
        return
    print(f"\nFACTS ({len(rec['facts'])})")
    for f in rec["facts"]:
        q = f"  [{f['qualifiers']}]" if f["qualifiers"] else ""
        print(f"  {f['subject']} -{f['predicate']}-> {f['object']}{q}")
        print(f"      \"{' '.join(f['quote'].split())[:100]}\"")
    print("\nSUMMARY\n" + rec["summary"])
    for c in rec["cells"]:
        print(f"\n[{c['entity']}]\n{c['text']}")

## 5 · The first unit, in full

In [ ]:
records = [derive(units[0])]
show(records[0], full=True)

## 6 · The rest of the document
Every unit prints its cast as it lands. Watching the same names recur under slightly
different forms is the point: that pile is what reconciliation is for.

In [ ]:
for unit in units[1:]:
    records.append(derive(unit))
    show(records[-1])

print(f"\nderived: {len(records)} units, {sum(len(r['entities']) for r in records)} local entities, "
      f"{sum(len(r['facts']) for r in records)} facts, ${spend():.4f}")

## 7 · Reconcile the major entities, bottom up
Every entity from every unit starts as its own cluster. Cheap signals nominate
candidate pairs: a shared name or surface form, a fact saying one is the other, a shared
name word. The strongest pairs are judged first, in small batches, by the judge model,
each verdict seeing both clusters' accumulated dossiers. A yes merges the clusters, so
later, harder pairs are decided with more evidence. Two entities listed in the same
unit are never paired: that unit already called them distinct. A minor entity may
join a major one, but two minors are never paired with each other.

In [ ]:
locals_ = []
for ui, rec in enumerate(records):
    for e in rec["entities"]:
        facts = [f for f in rec["facts"] if f["subject"] == e["name"]]
        locals_.append({"id": len(locals_), "ui": ui, "unit": rec["unit"], "name": e["name"],
                        "kind": e["kind"], "named": e["named"], "major": e["salience"] == "major",
                        "surfaces": {s.casefold() for s in e["surface_forms"]} | {e["name"].casefold()},
                        "is_a": [f["object"] for f in facts if f["predicate"] == "is_a"],
                        "facts": [f"{f['predicate']} {f['object']}" + (f" [{f['qualifiers']}]" if f["qualifiers"] else "")
                                  for f in facts],
                        "relations": [f"{f['predicate']} {f['object']}" for f in facts if f["object_is_entity"]]})

parent = list(range(len(locals_)))          # union-find over local ids
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(a, b):
    parent[find(b)] = find(a)

def words(name):
    return {w for w in re.findall(r"[a-z]+", name.casefold()) if len(w) > 3}

pairs = {}
for a in locals_:
    for b in locals_:
        if a["id"] >= b["id"] or a["ui"] == b["ui"] or not (a["major"] or b["major"]):
            continue
        if a["surfaces"] & b["surfaces"]:
            score = 1.0
        elif any(x.casefold() in b["surfaces"] for x in a["is_a"]) or \
                any(x.casefold() in a["surfaces"] for x in b["is_a"]):
            score = 0.85
        elif words(a["name"]) & words(b["name"]):
            score = 0.5
        else:
            continue
        pairs[(a["id"], b["id"])] = score
print(f"{len(locals_)} local entities, {sum(l['major'] for l in locals_)} major, {len(pairs)} candidate pairs")

In [ ]:
JUDGE_PROMPT = """Each PAIR below shows two entity clusters built from different units of one document. Decide for each pair whether the two clusters describe the SAME individual thing. Weigh all the evidence: shared names and forms, what each is said to be, its facts, and above all who and what it relates to; a fact stating that one is the other is near-decisive. Kind labels are per-unit guesses and often differ for the same individual, so never decide on them alone. Answer unsure only when the evidence genuinely cannot settle it.

Return JSON {"verdicts": [{"pair": n, "verdict": "same" | "different" | "unsure", "reason": "one sentence"}]}

"""

def members(root):
    return [l for l in locals_ if find(l["id"]) == root]

def dossier(root):
    ks = members(root)
    take = lambda key, n: sorted({x for l in ks for x in l[key]})[:n]
    return "\n".join([
        f"names: {', '.join(sorted({l['name'] for l in ks}))}",
        f"forms: {', '.join(sorted({s for l in ks for s in l['surfaces']})[:12])}",
        f"kinds: {', '.join(sorted({l['kind'] for l in ks}))}",
        f"is: {', '.join(take('is_a', 6)) or '(nothing stated)'}",
        f"facts: {'; '.join(take('facts', 10)) or '(none)'}",
        f"relations: {'; '.join(take('relations', 10)) or '(none)'}",
        f"units: {', '.join(sorted({l['unit'] for l in ks}))}"])

def judge(batch):
    """One call over up to ten pairs of cluster roots; returns verdicts in batch order."""
    prompt = JUDGE_PROMPT + "\n\n".join(f"PAIR {k}\nA:\n{dossier(ra)}\n\nB:\n{dossier(rb)}"
                                        for k, (ra, rb) in enumerate(batch, 1))
    out = generate(prompt, "judge", model=JUDGE, effort="medium")
    verdicts = {}
    for v in out["verdicts"]:
        try:
            verdicts[int(v["pair"]) - 1] = (v["verdict"], v["reason"])
        except (KeyError, TypeError, ValueError):
            pass
    return verdicts

ledger, different, deferred = [], set(), []
def decide(batch, verdicts, final=False):
    for k, (ra, rb) in enumerate(batch):
        verdict, reason = verdicts.get(k, ("unsure", "no verdict returned"))
        ledger.append({"a": dossier(ra).split("\n")[0][7:], "b": dossier(rb).split("\n")[0][7:],
                       "verdict": verdict, "reason": reason})
        if verdict == "same":
            union(ra, rb)
        elif verdict == "different" or final:
            different.add(frozenset((find(ra), find(rb))))
        else:
            deferred.append((ra, rb))

queue = sorted(pairs, key=lambda p: -pairs[p])
i = 0
while i < len(queue):
    batch = []
    while i < len(queue) and len(batch) < 10:
        ra, rb = find(queue[i][0]), find(queue[i][1])
        i += 1
        if ra != rb and frozenset((ra, rb)) not in different and (ra, rb) not in batch:
            batch.append((ra, rb))
    if batch:
        decide(batch, judge(batch))

# deferred pairs get one last look against the finished clusters; the default is to stay split
final = [(find(a), find(b)) for a, b in deferred if find(a) != find(b)]
for j in range(0, len(final), 10):
    decide(final[j:j + 10], judge(final[j:j + 10]), final=True)

roots = {find(l["id"]) for l in locals_}
major_roots = {find(l["id"]) for l in locals_ if l["major"]}
print(f"{sum(l['major'] for l in locals_)} major locals -> {len(major_roots)} major entities "
      f"({len(roots) - len(major_roots)} minor-only left unit-local); "
      f"{sum(1 for e in ledger if e['verdict'] == 'same')} merges, "
      f"{sum(1 for e in ledger if e['verdict'] == 'different')} kept apart, "
      f"{len(ledger)} verdicts, judge cost ${sum(c['cost'] for c in calls if c['stage'] == 'judge'):.3f}")

In [ ]:
def canonical(ks):
    """The cluster's name: the most-used proper name, longest on ties; else the most-used name."""
    named = [l for l in ks if l["named"]] or ks
    counts = {}
    for l in named:
        counts[l["name"]] = counts.get(l["name"], 0) + 1
    return max(counts, key=lambda n: (counts[n], len(n)))

entities = []
for root in roots:
    ks = members(root)
    entities.append({"name": canonical(ks), "major": root in major_roots,
                     "kinds": sorted({l["kind"] for l in ks}),
                     "units": sorted({l["unit"] for l in ks}),
                     "names": sorted({l["name"] for l in ks}), "members": [l["id"] for l in ks]})
entities.sort(key=lambda e: (not e["major"], -len(e["members"])))

majors_out = [e for e in entities if e["major"]]
print(f"MAJOR ENTITIES ({len(majors_out)})")
for e in majors_out:
    print(f"  {e['name']:<34} {'/'.join(e['kinds']):<18} {len(e['units']):>2} units   "
          f"{' | '.join(n for n in e['names'] if n != e['name'])[:70]}")
print(f"\nMERGE LEDGER ({len(ledger)})")
for v in ledger:
    print(f"  {v['verdict']:<9} {v['a'][:30]:<30} ~ {v['b'][:30]:<30} {v['reason'][:70]}")

## 8 · The store
Nodes are the major entities. Every fact whose subject resolved to a major entity is kept
under that node: an object that is another major entity is an edge; any other object, a
minor entity or a literal, is a property. Minor entities are not nodes and have no id, and
a fact between two minors is dropped here; both stay in the unit records as context.

In [ ]:
local_of = {(l["ui"], l["name"]): l["id"] for l in locals_}
node_of_local = {}
for e in entities:
    for m in e["members"]:
        node_of_local[m] = e["name"]
nodes = {e["name"]: {"name": e["name"], "kinds": e["kinds"], "units": e["units"], "names": e["names"],
                     "facts": []} for e in entities if e["major"]}

for ui, rec in enumerate(records):
    for f in rec["facts"]:
        s = node_of_local.get(local_of.get((ui, f["subject"])))
        if s not in nodes:
            continue
        o = node_of_local.get(local_of.get((ui, f["object"]))) if f["object_is_entity"] else None
        nodes[s]["facts"].append({"subject": s, "surface_predicate": f["predicate"], "predicate": f["predicate"],
                                  "object": o if o in nodes else f["object"], "object_is_node": o in nodes,
                                  "qualifiers": f["qualifiers"], "unit": rec["unit"], "quote": f["quote"]})

store_facts = [f for n in nodes.values() for f in n["facts"]]
n_edges = sum(1 for f in store_facts if f["object_is_node"] and f["object"] != f["subject"])
print(f"{len(nodes)} nodes, {len(store_facts)} facts ({n_edges} edges, {len(store_facts) - n_edges} properties); "
      f"{sum(len(r['facts']) for r in records) - len(store_facts)} facts about minor entities left in the unit records")

## 9 · Consolidate the predicates, flag the forks
The model named relations as each unit put them, so the vocabulary has diverged. Over
the store's facts: every surface predicate is tallied with its sample objects and
qualifiers, and one judge call groups the names that mean the same relation under a
canonical name. A single name used for two clearly different relations is a fork
candidate: flagged with its two senses, never split here. Facts keep the predicate the
model wrote and gain the canonical one.

In [ ]:
uses, objects, quals = {}, {}, {}
for f in store_facts:
    p = f["surface_predicate"]
    uses[p] = uses.get(p, 0) + 1
    objects.setdefault(p, [])
    if f["object"] not in objects[p]:
        objects[p].append(f["object"])
    if f["qualifiers"]:
        quals.setdefault(p, [])
        if f["qualifiers"] not in quals[p]:
            quals[p].append(f["qualifiers"])

listing = "\n".join(f"- {p} ({uses[p]}): objects {', '.join(objects[p][:5])}"
                    + (f"; qualifiers {', '.join(quals[p][:3])}" if p in quals else "")
                    for p in sorted(uses, key=lambda p: -uses[p]))

PREDICATE_PROMPT = f"""Below are the relation names used in facts about the main entities of one document, each with its use count, sample objects, and sample qualifiers. Two names express the same relation when swapping one for the other would not change what a fact claims; then group them under one canonical name, preferring the most-used member. Names that differ in meaning, direction, or specificity stay separate; a name that groups with nothing is its own group. Separately, a single name used for two clearly different relations, visible when its objects split into different kinds of thing, is a fork candidate: report it with its two senses.

Return JSON {{"groups": [{{"canonical": "...", "members": ["...", "..."]}}], "forks": [{{"predicate": "...", "senses": ["...", "..."], "why": "one sentence"}}]}}

RELATIONS:
{listing}"""

out = generate(PREDICATE_PROMPT, "predicates", model=JUDGE, effort="medium")
canonical_of = {}
for g in out["groups"]:
    for m in g["members"]:
        if m in uses:
            canonical_of[m] = g["canonical"]
forks = [fk for fk in out["forks"] if fk.get("predicate") in uses]
for f in store_facts:
    f["predicate"] = canonical_of.get(f["surface_predicate"], f["surface_predicate"])
edges = [f for f in store_facts if f["object_is_node"] and f["object"] != f["subject"]]

canon_uses = {}
for p, n in uses.items():
    canon_uses.setdefault(canonical_of.get(p, p), {})[p] = n
print(f"{len(uses)} surface predicates over {len(store_facts)} facts -> {len(canon_uses)} canonical\n")
print("VOCABULARY (canonical <- surface forms)")
for c, ms in sorted(canon_uses.items(), key=lambda kv: -sum(kv[1].values()))[:40]:
    print(f"  {c:<26} {sum(ms.values()):>4}   " + ", ".join(f"{m} ({n})" for m, n in sorted(ms.items(), key=lambda kv: -kv[1]) if m != c))
print(f"\nFORK CANDIDATES ({len(forks)})")
for fk in forks:
    print(f"  {fk['predicate']:<26} {' / '.join(fk['senses'])}   {fk.get('why', '')[:80]}")

print("\nNODES")
for n in sorted(nodes.values(), key=lambda n: -len(n["facts"])):
    props = list(dict.fromkeys(f"{f['predicate']} {f['object']}" for f in n["facts"] if not f["object_is_node"]))
    links = list(dict.fromkeys(f"{f['predicate']} {f['object']}" for f in n["facts"] if f["object_is_node"]))
    print(f"{n['name']}  ({len(n['units'])} units, {len(n['facts'])} facts, {len(links)} distinct edges, {len(props)} distinct properties)")
    print(f"    edges: {'; '.join(links[:6]) or '(none)'}")
    print(f"    properties: {'; '.join(props[:6]) or '(none)'}")

## 10 · The graph
Major entities as nodes, sized by how many units they appear in; an edge for every pair
that shares a fact, labeled with one of its predicates.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
for n in nodes.values():
    G.add_node(n["name"], size=len(n["units"]))
for e in edges:
    if G.has_edge(e["subject"], e["object"]):
        G[e["subject"]][e["object"]]["labels"].add(e["predicate"])
    else:
        G.add_edge(e["subject"], e["object"], labels={e["predicate"]})

pos = nx.spring_layout(G, seed=7, k=1.6)
plt.figure(figsize=(16, 11))
nx.draw_networkx_nodes(G, pos, node_size=[300 + 160 * G.nodes[n]["size"] for n in G], node_color="#cfe3f7")
nx.draw_networkx_edges(G, pos, alpha=0.4)
nx.draw_networkx_labels(G, pos, font_size=9)
nx.draw_networkx_edge_labels(G, pos, font_size=7,
                             edge_labels={(a, b): sorted(d["labels"])[0] for a, b, d in G.edges(data=True)})
plt.axis("off")
plt.savefig("graph.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges; saved graph.png")

## 11 · Cost and records

In [ ]:
by_stage = {}
for c in calls:
    by_stage[c["stage"]] = by_stage.get(c["stage"], 0) + c["cost"]
for stage, cost in by_stage.items():
    print(f"{stage:<10} ${cost:.4f}")
print(f"document cost ${spend():.4f} over {len(calls)} calls")

json.dump({"document": DOC, "units": records, "nodes": list(nodes.values()), "edges": edges,
           "entities": entities, "ledger": ledger, "predicates": canonical_of, "forks": forks,
           "calls": calls},
          open("store.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("saved store.json")

## 12 · Abstracts: one summary per major entity, then the document

In [ ]:
NARRATOR = JUDGE            # the visible layer runs on the strong model; MODEL for a cheaper pass

def entity_summary(n):
    facts = list(dict.fromkeys(f"{f['predicate']} {f['object']}" + (f" [{f['qualifiers']}]" if f["qualifiers"] else "")
                               for f in n["facts"]))
    cells = [f"[{u['unit']}] {c['text']}" for u in records for c in u["cells"] if c["entity"] in n["names"]]
    prompt = f"""Below are the accumulated records for one entity of a longer document: its facts, then its cells in reading order. Write one comprehensive summary of this entity: what it is, its attributes and relationships, and the course of what it does and undergoes across the units. Ground every statement only in these records; use no outside knowledge and no names that do not appear below. At most 200 words.

Return JSON {{"summary": "..."}}

ENTITY: {n['name']} ({', '.join(n['kinds'])})

FACTS:
{chr(10).join(facts) or '(none)'}

CELLS (reading order):
{chr(10).join(cells) or '(none)'}"""
    return generate(prompt, "abstract", model=NARRATOR)["summary"]

abstracts = {}
for n in sorted(nodes.values(), key=lambda n: -len(n["units"])):
    abstracts[n["name"]] = entity_summary(n)
    print(f"\n{n['name']}  ({len(n['units'])} units)\n{abstracts[n['name']]}")

unit_summaries = "\n".join(f"[{u['unit']}] {u['summary']}" for u in records)
DOC_PROMPT = f"""Below are the unit summaries of one document, in reading order. Write one summary of the whole document in 8-12 sentences: what it is about, who and what matter most, and how it unfolds from beginning to end. Ground every statement only in these summaries; use no outside knowledge.

Return JSON {{"summary": "..."}}

UNIT SUMMARIES:
{unit_summaries}"""
document_summary = generate(DOC_PROMPT, "abstract", model=NARRATOR)["summary"]
print(f"\nDOCUMENT\n{document_summary}")